# Session 5: Agentic AI with LangChain Agents
**Duration:** 1.5 Hours | **Day 2** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand what Agentic AI means and how it differs from simple LLM calls
2. Build custom tools that LLM agents can use
3. Create a tool-using AI agent with LangChain
4. Understand the ReAct (Reasoning + Acting) pattern
5. Build an agent that chains multiple tools together to solve problems

---

## Prerequisites
- Completed Day 1 sessions
- Ollama running with `llama3.2`
- Packages: `pip install langchain langchain-ollama langchain-community`

---
## Part 1: What is Agentic AI? (Theory - 20 min)

### 1.1 From Chatbots to Agents

```
Level 0: Simple LLM Call
  User --> LLM --> Response
  (No tools, no memory, single turn)

Level 1: LLM with Tools (Function Calling)
  User --> LLM --> Decides to use tool --> Tool executes --> LLM formats response
  (Single tool use, one step)

Level 2: ReAct Agent (Reasoning + Acting)
  User --> LLM --> Think --> Act (use tool) --> Observe result --> Think again --> Act again --> Final Answer
  (Multi-step reasoning, tool chaining)

Level 3: Autonomous Agent
  User --> Agent --> Plans steps --> Executes --> Evaluates --> Replans if needed --> Final Answer
  (Planning, memory, self-correction)

Level 4: Multi-Agent System
  User --> Orchestrator --> Agent 1 (Research) --> Agent 2 (Analyze) --> Agent 3 (Review) --> Final Output
  (Multiple specialized agents collaborating)
```

### 1.2 The ReAct Pattern

ReAct (Reasoning + Acting) is the most common agent pattern:

```
LOOP:
  1. THOUGHT: "I need to find out X to answer this question"
  2. ACTION: Use tool Y with parameters Z
  3. OBSERVATION: Tool returned this result
  4. THOUGHT: "Now I know X, but I also need W"
  5. ACTION: Use tool A with parameters B
  6. OBSERVATION: Tool returned this result
  7. THOUGHT: "I now have all the information needed"
  8. FINAL ANSWER: Here is the complete answer
```

### 1.3 Key Components of an Agent

| Component | Description | Example |
|-----------|-------------|--------|
| **LLM (Brain)** | Decides what to do next | llama3.2, GPT-4 |
| **Tools** | Actions the agent can take | Calculator, search, database query |
| **Memory** | Past interactions and context | Conversation history |
| **Planning** | Strategy for solving the task | Step-by-step decomposition |
| **Observation** | Results from tool execution | API responses, calculations |

---
## Part 2: Building Custom Tools (Hands-On - 20 min)

Before building an agent, we need tools for it to use.

In [ ]:
# 2.1 Setup
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
import json
import math
import requests
from datetime import datetime

# Initialize the LLM
llm = ChatOllama(model="llama3.2", temperature=0)

print("LLM initialized!")

In [ ]:
# 2.2 Tool 1: Advanced Calculator

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic arithmetic,
    power (**), sqrt, sin, cos, tan, log, pi, e.

    Examples: '2 + 3 * 4', 'sqrt(144)', '2 ** 10', 'sin(pi/2)'

    Args:
        expression: A mathematical expression to evaluate
    """
    # Safe math evaluation
    allowed_names = {
        "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
        "tan": math.tan, "log": math.log, "log10": math.log10,
        "pi": math.pi, "e": math.e, "abs": abs, "pow": pow,
        "round": round
    }
    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {expression} = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {str(e)}"

# Test the tool
print(calculator.invoke("2 ** 10"))
print(calculator.invoke("sqrt(144) + 5"))
print(calculator.invoke("sin(pi/2)"))

In [ ]:
# 2.3 Tool 2: Web Search (Mock -- in production, use a real search API)

@tool
def web_search(query: str) -> str:
    """Search the web for information. Returns relevant search results.
    Use this when you need up-to-date information or facts you don't know.

    Args:
        query: The search query string
    """
    # Mock search results (in production, use SerpAPI, Tavily, or similar)
    mock_results = {
        "population of india": "India's population as of 2024 is approximately 1.44 billion, making it the most populous country in the world.",
        "python programming": "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It is known for its simple syntax and is widely used in data science, AI, web development, and automation.",
        "langchain framework": "LangChain is an open-source framework for building applications with large language models. It provides tools for chains, agents, memory, and retrieval augmented generation.",
        "ollama": "Ollama is a tool for running open-source large language models locally. It supports models like LLaMA, Mistral, and Gemma. Available at ollama.com.",
        "tumkur karnataka": "Tumkur (Tumakuru) is a city in Karnataka, India, located about 70 km northwest of Bangalore. It is known for educational institutions including Sidganga Institute of Technology.",
    }

    query_lower = query.lower()
    for key, value in mock_results.items():
        if any(word in query_lower for word in key.split()):
            return f"Search results for '{query}':\n{value}"

    return f"Search results for '{query}':\nNo specific results found. The query may require a more specific search term."

# Test
print(web_search.invoke("What is the population of India?"))

In [ ]:
# 2.4 Tool 3: Date/Time Tool

@tool
def get_current_datetime(format: str = "full") -> str:
    """Get the current date and time.

    Args:
        format: 'full' for complete datetime, 'date' for date only, 'time' for time only
    """
    now = datetime.now()
    if format == "date":
        return f"Current date: {now.strftime('%Y-%m-%d (%A)')}"
    elif format == "time":
        return f"Current time: {now.strftime('%H:%M:%S')}"
    else:
        return f"Current datetime: {now.strftime('%Y-%m-%d %H:%M:%S (%A)')}"

# Test
print(get_current_datetime.invoke("full"))

In [ ]:
# 2.5 Tool 4: Unit Converter

@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between common units of measurement.

    Supported conversions:
    - Temperature: celsius, fahrenheit, kelvin
    - Length: meters, feet, inches, km, miles
    - Weight: kg, pounds, grams, ounces

    Args:
        value: The numeric value to convert
        from_unit: The source unit
        to_unit: The target unit
    """
    conversions = {
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
        ("celsius", "kelvin"): lambda x: x + 273.15,
        ("kelvin", "celsius"): lambda x: x - 273.15,
        ("meters", "feet"): lambda x: x * 3.28084,
        ("feet", "meters"): lambda x: x / 3.28084,
        ("km", "miles"): lambda x: x * 0.621371,
        ("miles", "km"): lambda x: x / 0.621371,
        ("kg", "pounds"): lambda x: x * 2.20462,
        ("pounds", "kg"): lambda x: x / 2.20462,
    }

    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    else:
        return f"Conversion from {from_unit} to {to_unit} is not supported."

# Test
print(unit_converter.invoke({"value": 100, "from_unit": "celsius", "to_unit": "fahrenheit"}))
print(unit_converter.invoke({"value": 5, "from_unit": "km", "to_unit": "miles"}))

---
## Part 3: Creating a ReAct Agent (Hands-On - 25 min)

Now we bind our tools to the LLM and create an agent that can reason about which tools to use.

In [ ]:
# 3.1 Bind tools to the LLM
# This tells the LLM about available tools and their schemas

tools = [calculator, web_search, get_current_datetime, unit_converter]

# Bind tools -- the LLM will now know about these tools
llm_with_tools = llm.bind_tools(tools)

print("Tools bound to LLM:")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}...")

In [ ]:
# 3.2 Test tool calling -- the LLM decides which tool to call

response = llm_with_tools.invoke("What is 25 * 47 + 13?")

print("LLM Response:")
print(f"  Content: {response.content}")
print(f"  Tool Calls: {response.tool_calls}")

if response.tool_calls:
    for tc in response.tool_calls:
        print(f"\n  Tool: {tc['name']}")
        print(f"  Args: {tc['args']}")

In [ ]:
# 3.3 Build a complete agent with the agent loop
# This implements the full ReAct cycle: Think --> Act --> Observe --> Repeat

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_agent(user_query: str, max_iterations: int = 5):
    """Run the ReAct agent loop."""
    print(f"\n{'='*60}")
    print(f"USER: {user_query}")
    print(f"{'='*60}")

    # Initialize conversation
    messages = [
        SystemMessage(content="""You are a helpful AI assistant with access to tools.
        Always use tools when they can help answer the question accurately.
        Think step by step about what information you need.
        After using tools, provide a clear final answer."""),
        HumanMessage(content=user_query)
    ]

    # Tool lookup dictionary
    tool_map = {t.name: t for t in tools}

    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        # Get LLM response
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If no tool calls, we have the final answer
        if not response.tool_calls:
            print(f"\nFINAL ANSWER: {response.content}")
            return response.content

        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            print(f"  THOUGHT: I need to use {tool_name}")
            print(f"  ACTION: {tool_name}({tool_args})")

            # Execute the tool
            if tool_name in tool_map:
                result = tool_map[tool_name].invoke(tool_args)
            else:
                result = f"Error: Unknown tool '{tool_name}'"

            print(f"  OBSERVATION: {result}")

            # Add tool result to messages
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

    return "Agent reached maximum iterations without a final answer."

print("Agent ready!")

In [ ]:
# 3.4 Test the agent with different queries

# Query 1: Simple calculation
run_agent("What is the square root of 2025?")

In [ ]:
# Query 2: Multi-tool query
run_agent("What is today's date, and what is 365 * 24 * 60 (minutes in a year)?")

In [ ]:
# Query 3: Web search + computation
run_agent("Search for information about Tumkur, Karnataka")

In [ ]:
# Query 4: Unit conversion
run_agent("Convert 37 degrees Celsius to Fahrenheit. Is that considered a fever for a human body?")

---
## Part 4: Using LangGraph for Stateful Agents (Hands-On - 15 min)

LangGraph provides a more robust way to build agents with proper state management.

In [ ]:
# 4.1 LangGraph-style agent (simplified version)
# LangGraph uses a graph-based approach to define agent workflows

# For this workshop, we'll build a structured agent with memory

class StructuredAgent:
    """A structured agent with memory and tool access."""

    def __init__(self, llm, tools, system_prompt=""):
        self.llm = llm.bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.system_prompt = system_prompt
        self.memory = []  # Conversation history
        self.tool_usage_log = []  # Track all tool calls

    def reset(self):
        """Clear memory and logs."""
        self.memory = []
        self.tool_usage_log = []

    def run(self, query: str, verbose: bool = True) -> str:
        """Run the agent on a query."""
        # Build messages
        messages = []
        if self.system_prompt:
            messages.append(SystemMessage(content=self.system_prompt))
        messages.extend(self.memory)
        messages.append(HumanMessage(content=query))

        iteration = 0
        while iteration < 10:
            iteration += 1
            response = self.llm.invoke(messages)
            messages.append(response)

            if not response.tool_calls:
                # Final answer
                self.memory.append(HumanMessage(content=query))
                self.memory.append(AIMessage(content=response.content))

                if verbose:
                    print(f"Agent completed in {iteration} iteration(s)")
                    print(f"Tools used: {len(self.tool_usage_log)} call(s)")

                return response.content

            # Execute tool calls
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                if verbose:
                    print(f"  [{iteration}] Calling: {tool_name}({tool_args})")

                if tool_name in self.tools:
                    result = self.tools[tool_name].invoke(tool_args)
                else:
                    result = f"Error: Tool '{tool_name}' not found"

                self.tool_usage_log.append({
                    "tool": tool_name,
                    "args": tool_args,
                    "result": str(result)
                })

                messages.append(ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                ))

        return "Max iterations reached"

# Create the agent
agent = StructuredAgent(
    llm=llm,
    tools=tools,
    system_prompt="""You are a helpful AI assistant with access to tools.
    Use tools to provide accurate answers. Think step by step.
    Remember previous conversations with the user."""
)

print("Structured Agent created!")

In [ ]:
# 4.2 Test the structured agent with multi-turn conversation

# Turn 1
print("=== Turn 1 ===")
answer1 = agent.run("What is 15% of 8500?")
print(f"Answer: {answer1}")

print("\n" + "="*40)

# Turn 2 (the agent remembers the previous conversation)
print("\n=== Turn 2 ===")
answer2 = agent.run("Now calculate double of that result")
print(f"Answer: {answer2}")

In [ ]:
# 4.3 View the agent's tool usage log

print("=== Agent Tool Usage Log ===")
for i, log in enumerate(agent.tool_usage_log, 1):
    print(f"\nCall {i}:")
    print(f"  Tool: {log['tool']}")
    print(f"  Args: {log['args']}")
    print(f"  Result: {log['result']}")

---
## Part 5: Building a Practical Agent -- Student Helper (Hands-On - 10 min)

Let's combine everything into a practical student helper agent.

In [ ]:
# 5.1 Student-specific tools

@tool
def gpa_calculator(grades: str) -> str:
    """Calculate GPA from a comma-separated list of VTU grades.
    Grade scale: S=10, A=9, B=8, C=7, D=6, E=5, F=0

    Args:
        grades: Comma-separated grades like 'S,A,B,A,S,C'
    """
    grade_points = {"S": 10, "A": 9, "B": 8, "C": 7, "D": 6, "E": 5, "F": 0}
    grade_list = [g.strip().upper() for g in grades.split(",")]

    invalid = [g for g in grade_list if g not in grade_points]
    if invalid:
        return f"Invalid grades: {invalid}. Valid: S, A, B, C, D, E, F"

    points = [grade_points[g] for g in grade_list]
    gpa = sum(points) / len(points)

    classification = (
        "First Class with Distinction" if gpa >= 7.5 else
        "First Class" if gpa >= 6.5 else
        "Second Class" if gpa >= 5.5 else
        "Pass" if gpa >= 4.0 else "Fail"
    )

    return f"Grades: {grade_list}\nGPA: {gpa:.2f}/10\nClassification: {classification}"


@tool
def attendance_checker(classes_attended: int, total_classes: int) -> str:
    """Check if a student meets the 75% minimum attendance requirement.

    Args:
        classes_attended: Number of classes attended
        total_classes: Total number of classes held
    """
    if total_classes == 0:
        return "Error: Total classes cannot be zero"

    percentage = (classes_attended / total_classes) * 100
    eligible = percentage >= 75

    result = f"Attendance: {classes_attended}/{total_classes} = {percentage:.1f}%\n"
    result += f"Minimum required: 75%\n"
    result += f"Status: {'ELIGIBLE for exams' if eligible else 'NOT ELIGIBLE for exams'}\n"

    if not eligible:
        needed = math.ceil(0.75 * total_classes) - classes_attended
        result += f"Classes needed to reach 75%: {needed} more (assuming no new classes)"

    return result


@tool
def fee_calculator(quota: str, years: int, hostel: bool) -> str:
    """Calculate total B.Tech fees at SIT.

    Args:
        quota: One of 'government', 'comedk', or 'management'
        years: Number of years (1-4)
        hostel: Whether hostel is included
    """
    fees = {"government": 55000, "comedk": 120000, "management": 250000}
    if quota.lower() not in fees:
        return f"Invalid quota. Options: government, comedk, management"

    tuition = fees[quota.lower()] * years
    hostel_fee = 75000 * years if hostel else 0
    total = tuition + hostel_fee

    return f"Fee Breakdown:\n  Tuition ({quota}): Rs {tuition:,}\n  Hostel: Rs {hostel_fee:,}\n  Total for {years} years: Rs {total:,}"


# Create student helper agent
student_tools = [calculator, gpa_calculator, attendance_checker, fee_calculator, get_current_datetime]

student_agent = StructuredAgent(
    llm=llm,
    tools=student_tools,
    system_prompt="""You are a helpful student advisor for Sidganga Institute of Technology (SIT), Tumkur.
    You help students with GPA calculations, attendance tracking, fee queries, and general academic advice.
    Always use the available tools to provide accurate calculations.
    Be friendly and supportive."""
)

print("Student Helper Agent created with tools:")
for t in student_tools:
    print(f"  - {t.name}")

In [ ]:
# 5.2 Test the student agent

print("=" * 50)
result = student_agent.run("My grades this semester are S, A, B, A, S, B, A, C. What is my GPA?")
print(f"\nAnswer: {result}")

In [ ]:
student_agent.reset()
print("=" * 50)
result = student_agent.run("I have attended 58 out of 80 classes. Am I eligible for exams?")
print(f"\nAnswer: {result}")

In [ ]:
student_agent.reset()
print("=" * 50)
result = student_agent.run("Calculate total fees for 4 years under COMEDK quota with hostel")
print(f"\nAnswer: {result}")

---
## Exercises

### Exercise 1: Add a New Tool
Create a `placement_info` tool that returns placement statistics for a given department. Add it to the student agent.

### Exercise 2: Complex Query
Ask the agent: "If I have a GPA of 8.5 and I'm in CSE, what are my chances of getting placed? Also, calculate the total fee for 4 years under government quota with hostel."

### Exercise 3: Tool Chain
Create a scenario where the agent needs to use 3+ tools in sequence to answer a single question.

---

## Summary

### Key Takeaways:
1. **Agents** go beyond simple LLM calls by reasoning about which tools to use
2. **ReAct pattern**: Think --> Act --> Observe --> Repeat until answer is found
3. **Tools** are Python functions with clear descriptions that LLMs can call
4. **Type hints and docstrings** are critical -- they tell the LLM how to use tools
5. **Memory** allows agents to maintain context across multiple turns
6. Agents can **chain multiple tools** to solve complex problems

### Next Session: MCP Deep Dive + Agentic Integration
We will connect these agents to MCP servers, enabling standardized tool access across applications.